# Exploring `CsvSeparatorDetector`

This notebook is a companion for interactively poking at how `CsvSeparatorDetector` (in `csvseparator/separator.py`) analyzes a CSV file: column profiles, structural-issue detection, and quote-fix repair, step by step.

It is **not** part of the CLI contract described in the README/CLAUDE.md — `main.py`'s `run_interactive`/`run_noninteractive` are still the only supported entry points for actually running the tool. This is a dev/exploration aid on top of the same library code.

Requires the `notebook` extra: `pip install -e ".[notebook]"`.

In [1]:
from pathlib import Path
from csvseparator.separator import CsvSeparatorDetector

sample_path = Path("data/messy_sample.csv")
print(sample_path.read_text(encoding="utf-8"))

detector = CsvSeparatorDetector(str(sample_path))
detector.get_file_info()

id,name,address,amount,notes
1,Alice,"123 Main St, Suite 100",25.00,Prefers email
2,Bob,456 Oak Ave, Suite 200, Metropolis,54.50,Prefers evening drop-off
3,Carol,"789 Pine Rd, Apt 4",12.25,ok



{'total_rows': 4,
 'header_row': ['id', 'name', 'address', 'amount', 'notes'],
 'column_count': 5,
 'file_size': 192}

## Parse warnings

Non-fatal issues encountered while reading the file: encoding fallback (`utf-8-sig` → `cp1252`) or a lenient re-parse after a strict `csv.Error`.

In [2]:
detector.get_parse_warnings()

[]

## Structural issue detection

Rows that look inconsistent with the rest of the file — the same check the CLI prints to the screen. Row 2 (Bob) has an unescaped comma in `address`, so it shows up with extra columns.

In [3]:
report = detector.get_column_mismatch_report()
report

[{'row_number': 3,
  'expected_columns': 5,
  'actual_columns': 7,
  'reasons': ['expected 5 columns but found 7',
   "column 4 has text-like value 'Suite 200' but this column is usually numeric",
   'extra values detected: 54.50, Prefers evening drop-off'],
  'highlighted_row': "❗ Row 3: expected 5 columns but found 7; column 4 has text-like value 'Suite 200' but this column is usually numeric; extra values detected: 54.50, Prefers evening drop-off -> [2] | [Bob] | [456 Oak Ave] | [ Suite 200] | [ Metropolis] | [54.50] | [Prefers evening drop-off]"}]

In [4]:
for issue in report:
    print(issue["highlighted_row"])

❗ Row 3: expected 5 columns but found 7; column 4 has text-like value 'Suite 200' but this column is usually numeric; extra values detected: 54.50, Prefers evening drop-off -> [2] | [Bob] | [456 Oak Ave] | [ Suite 200] | [ Metropolis] | [54.50] | [Prefers evening drop-off]


## Quote-fix repair

`repair_quoting()` tries to identify the single column whose unescaped comma(s) produced the extra tokens, by checking which candidate merge leaves no remaining type/empty mismatches against the column's profile. Returns the full repaired row set plus a per-row summary.

In [5]:
repaired_rows, summary = detector.repair_quoting()
summary

[{'row_number': 3, 'status': 'repaired', 'quoted_column': 'address'}]

In [6]:
repaired_rows

[['id', 'name', 'address', 'amount', 'notes'],
 ['1', 'Alice', '123 Main St, Suite 100', '25.00', 'Prefers email'],
 ['2',
  'Bob',
  '456 Oak Ave, Suite 200, Metropolis',
  '54.50',
  'Prefers evening drop-off'],
 ['3', 'Carol', '789 Pine Rd, Apt 4', '12.25', 'ok']]

## Separator collisions

How many existing data values already contain the separator you're about to switch to. This is a warning, not a blocker — the output CSV is still valid, values just get quoted.

In [7]:
detector.count_separator_collisions("|")

0

## Converting

`convert_separator` writes an actual output file — the same call the CLI makes once `--yes` is passed. Passing `quote_fix=True` repairs what it confidently can before writing.

In [8]:
output_path = detector.convert_separator("|", output_dir="scratch_out", quote_fix=True)
output_path

'scratch_out\\messy_sample_converted_pipe.csv'

In [9]:
print(Path(output_path).read_text(encoding="utf-8"))

id|name|address|amount|notes
1|Alice|123 Main St, Suite 100|25.00|Prefers email
2|Bob|456 Oak Ave, Suite 200, Metropolis|54.50|Prefers evening drop-off
3|Carol|789 Pine Rd, Apt 4|12.25|ok

